
### Понимание задачи
Даны два набора данных:
1. **Тренировочный набор** (второй `<DOCUMENT>` в запросе) содержит столбец `SalePrice` — это целевая переменная, которую нужно предсказать. Этот набор нужно разделить на обучающую (`train`) и тестовую (`test`) выборки для проверки модели.
2. **Тестовый набор** (первый `<DOCUMENT>` в запросе) не содержит `SalePrice`, и на нем нужно сделать предсказания для загрузки на Kaggle.

Задача требует:
- Провести предобработку данных (обработка пропусков, кодирование категориальных признаков, масштабирование и т.д.).
- Обучить линейную регрессию.
- Сравнить модели с регуляризацией (например, Ridge, Lasso) и без.
- Добиться результата лучше бейзлайна на Kaggle (RMSE < 0.16, согласно стандартным ожиданиям для этого набора данных).

### План действий
1. **Загрузка и анализ данных**:
   - Разделим тренировочный набор на `train` (80%) и `test` (20%).
   - Проведем первичный анализ: пропуски, распределение `SalePrice`, корреляции.
2. **Предобработка**:
   - Обработка пропусков (числовые и категориальные признаки).
   - Кодирование категориальных переменных (One-Hot Encoding, Label Encoding).
   - Масштабирование числовых признаков.
   - Удаление выбросов (опционально).
3. **Эксперименты**:
   - Базовая линейная регрессия.
   - Регрессия с регуляризацией (Ridge, Lasso).
   - Оценка на внутренней тестовой выборке (RMSE).
4. **Финальная модель**:
   - Обучение на полном тренировочном наборе.
   - Предсказание для тестового набора.
   - Подготовка файла для Kaggle.


### 0,01634 - Public, 0,01077 на Private

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# Загрузка данных
train_data = pd.read_csv('house-prices-hw/train_hw.csv') 
test_data = pd.read_csv('house-prices-hw/test_hw.csv') 

# Разделение на признаки и целевую переменную
X = train_data.drop(columns=['SalePrice', 'Id'])
y = train_data['SalePrice']

# Разделение на обучающую и тестовую выборки (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Функция предобработки данных
def impute_missing_values(df):
    numeric_cols = df.select_dtypes(exclude=['object']).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    
    # Категориальные признаки: заполняем "None"
    categorical_cols = df.select_dtypes(include=['object']).columns
    df[categorical_cols] = df[categorical_cols].fillna('None')
    
    return df

X_train = impute_missing_values(X_train.copy())
X_test = impute_missing_values(X_test.copy())
test_data_processed = impute_missing_values(test_data.drop(columns=['Id']).copy())

# Кодирование категориальных признаков (One-Hot Encoding)
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
test_data_encoded = pd.get_dummies(test_data_processed, drop_first=True)

# Выравнивание колонок
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)
X_train_encoded, test_data_encoded = X_train_encoded.align(test_data_encoded, join='left', axis=1, fill_value=0)
X_test_encoded, test_data_encoded = X_test_encoded.align(test_data_encoded, join='left', axis=1, fill_value=0)

# Логарифмирование целевой переменной
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Масштабирование числовых признаков
scaler = StandardScaler()
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
X_train_encoded[numeric_cols] = scaler.fit_transform(X_train_encoded[numeric_cols])
X_test_encoded[numeric_cols] = scaler.transform(X_test_encoded[numeric_cols])
test_data_encoded[numeric_cols] = scaler.transform(test_data_encoded[numeric_cols])

# # Визуализация распределения SalePrice
# plt.figure(figsize=(8, 6))
# sns.histplot(y_train, kde=True)
# plt.title('Распределение SalePrice')
# plt.show()

# plt.figure(figsize=(8, 6))
# sns.histplot(y_train_log, kde=True)
# plt.title('Распределение логарифма SalePrice')
# plt.show()

# Эксперимент 1: Базовая линейная регрессия
lr = LinearRegression()
lr.fit(X_train_encoded, y_train_log)
y_pred_log = lr.predict(X_test_encoded)
rmse_lr = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
print(f"RMSE базовой линейной регрессии: {rmse_lr:.4f}")

# Эксперимент 2: Ridge регрессия
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_encoded, y_train_log)
y_pred_ridge_log = ridge.predict(X_test_encoded)
rmse_ridge = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge_log))
print(f"RMSE Ridge регрессии: {rmse_ridge:.4f}")

# Эксперимент 3: Lasso регрессия
lasso = Lasso(alpha=0.001, max_iter=10000)
lasso.fit(X_train_encoded, y_train_log)
y_pred_lasso_log = lasso.predict(X_test_encoded)
rmse_lasso = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso_log))
print(f"RMSE Lasso регрессии: {rmse_lasso:.4f}")

# # Корреляция числовых признаков с SalePrice
# plt.figure(figsize=(12, 8))
# numeric_cols_with_target = numeric_cols.union(['SalePrice'])
# sns.heatmap(train_data[numeric_cols_with_target].corr(), annot=True, cmap='coolwarm', fmt='.2f')
# plt.title('Корреляция числовых признаков с SalePrice')
# plt.show()

# Финальная модель: обучение на полном тренировочном наборе
X_full = train_data.drop(columns=['SalePrice', 'Id'])
y_full = np.log1p(train_data['SalePrice'])
X_full_encoded = pd.get_dummies(impute_missing_values(X_full.copy()), drop_first=True)
X_full_encoded, test_data_encoded = X_full_encoded.align(test_data_encoded, join='left', axis=1, fill_value=0)
X_full_encoded[numeric_cols] = scaler.fit_transform(X_full_encoded[numeric_cols])

final_model = Lasso(alpha=0.001, max_iter=10000)
final_model.fit(X_full_encoded, y_full)

# Предсказания для тестового набора
test_pred_log = final_model.predict(test_data_encoded)
test_pred = np.expm1(test_pred_log)  # Обратное логарифмирование

# Подготовка файла для Kaggle
submission = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': test_pred})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

RMSE базовой линейной регрессии: 2.6072
RMSE Ridge регрессии: 0.2318
RMSE Lasso регрессии: 0.1410
Файл submission.csv успешно сохранен!


### 0.01549 - Public, 0,1291 - Private   
- Добавлен фича инжениринг 
- Oбработка пропусков для чилсенных призноков через KNN


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.model_selection import cross_val_score

# Загрузка данных
train_data = pd.read_csv('house-prices-hw/train_hw.csv')
test_data = pd.read_csv('house-prices-hw/test_hw.csv')
test_ids = test_data['Id']
train_data = train_data.drop(columns=['Id'])
test_data = test_data.drop(columns=['Id'])

# Создание новых признаков
def create_new_features(df):
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['HasPool'] = df['PoolArea'] > 0
    return df

train_data = create_new_features(train_data)
test_data = create_new_features(test_data)

# Определение числовых и категориальных колонок
numeric_cols = train_data.select_dtypes(exclude=['object']).columns.drop('SalePrice')
categorical_cols = train_data.select_dtypes(include=['object']).columns

# Обработка пропусков
def impute_missing_values(df, numeric_cols, categorical_cols):
    imputer = KNNImputer(n_neighbors=5)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    df[categorical_cols] = df[categorical_cols].fillna('None')
    return df

train_data = impute_missing_values(train_data, numeric_cols, categorical_cols)
test_data = impute_missing_values(test_data, numeric_cols, categorical_cols)

# Логарифмирование SalePrice
y = np.log1p(train_data['SalePrice'])
X = train_data.drop(columns=['SalePrice'])

# Удаление выбросов по log(SalePrice) с помощью IQR
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mask = (y >= lower_bound) & (y <= upper_bound)
X = X[mask]
y = y[mask]

# One-Hot Encoding для категориальных признаков
X_encoded = pd.get_dummies(X, drop_first=True)
test_encoded = pd.get_dummies(test_data, drop_first=True)

# Выравнивание колонок
X_encoded, test_encoded = X_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Масштабирование признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
test_scaled = scaler.transform(test_encoded)

# # Визуализация распределения SalePrice
# plt.figure(figsize=(8, 6))
# sns.histplot(np.expm1(y), kde=True)  # Обратное преобразование для отображения исходных цен
# plt.title('Распределение SalePrice')
# plt.show()

# plt.figure(figsize=(8, 6))
# sns.histplot(y, kde=True)
# plt.title('Распределение логарифма SalePrice')
# plt.show()

# # Корреляция числовых признаков с SalePrice
# plt.figure(figsize=(12, 8))
# numeric_cols_with_target = numeric_cols.union(['SalePrice'])
# sns.heatmap(train_data[numeric_cols_with_target].corr(), annot=True, cmap='coolwarm', fmt='.2f')
# plt.title('Корреляция числовых признаков с SalePrice')
# plt.show()

# Обучение Lasso с кросс-валидацией
lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1.0], cv=5)
lasso.fit(X_scaled, y)
lasso_scores = cross_val_score(lasso, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
lasso_rmse = np.sqrt(-lasso_scores)
print(f"Средний RMSE на кросс-валидации (Lasso): {lasso_rmse.mean():.4f}")

# Обучение Ridge с кросс-валидацией для сравнения
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5)
ridge.fit(X_scaled, y)
ridge_scores = cross_val_score(ridge, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
ridge_rmse = np.sqrt(-ridge_scores)
print(f"Средний RMSE на кросс-валидации (Ridge): {ridge_rmse.mean():.4f}")

# Предсказания на тестовом наборе (используем Lasso как основную модель)
y_pred_log = lasso.predict(test_scaled)
y_pred = np.expm1(y_pred_log)

# Сохранение результатов
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

C:\Users\maksc\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.026821833618830837, tolerance: 0.012200803599384176
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\maksc\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.01896788605193578, tolerance: 0.012200803599384176
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\maksc\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.11651318706091018, tolerance: 0.01201236471381194
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\maksc\AppData

Средний RMSE на кросс-валидации (Lasso): 0.1535
Средний RMSE на кросс-валидации (Ridge): 0.1628
Файл submission.csv успешно сохранен!


### 0.01438 - Public, 0,01198 - Private
- Добавлены новые фичи
- Логарифмирование признаков с сильной скошенностью
- Добавление полиномиальных признаков для ключевых числовых колонок


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.model_selection import cross_val_score

# Загрузка данных
train_data = pd.read_csv('house-prices-hw/train_hw.csv')
test_data = pd.read_csv('house-prices-hw/test_hw.csv')
test_ids = test_data['Id']
train_data = train_data.drop(columns=['Id'])
test_data = test_data.drop(columns=['Id'])

# Создание новых признаков
def create_new_features(df):
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['HasPool'] = df['PoolArea'] > 0
    df['OverallQualitySF'] = df['OverallQual'] * df['TotalSF']
    df['GarageScore'] = df['GarageCars'] * df['GarageArea']
    return df

train_data = create_new_features(train_data)
test_data = create_new_features(test_data)

# Определение числовых и категориальных колонок
numeric_cols = train_data.select_dtypes(exclude=['object']).columns.drop('SalePrice')
categorical_cols = train_data.select_dtypes(include=['object']).columns

def impute_missing_values(df, numeric_cols, categorical_cols):
    imputer = KNNImputer(n_neighbors=5)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    df[categorical_cols] = df[categorical_cols].fillna('None')
    categorical_imputer = SimpleImputer(strategy='most_frequent', fill_value='None')
    df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols])
    return df

train_data = impute_missing_values(train_data, numeric_cols, categorical_cols)
test_data = impute_missing_values(test_data, numeric_cols, categorical_cols)

# Логарифмирование признаков с сильной скошенностью
skewed_cols = ['TotalSF', 'GarageArea', '1stFlrSF', 'TotalBsmtSF']
for col in skewed_cols:
    train_data[col] = np.log1p(train_data[col])
    test_data[col] = np.log1p(test_data[col])

# Логарифмирование SalePrice
y = np.log1p(train_data['SalePrice'])
X = train_data.drop(columns=['SalePrice'])

# Удаление выбросов по log(SalePrice) с IQR 1.5
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mask = (y >= lower_bound) & (y <= upper_bound)
X = X[mask]
y = y[mask]

# One-Hot Encoding для категориальных признаков
X_encoded = pd.get_dummies(X, drop_first=True)
test_encoded = pd.get_dummies(test_data, drop_first=True)

# Выравнивание колонок
X_encoded, test_encoded = X_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Сохранение индексов ключевых числовых колонок для полиномизации
key_numeric_cols = ['OverallQual', 'TotalSF', 'GrLivArea']
numeric_indices = [X_encoded.columns.get_loc(col) for col in key_numeric_cols if col in X_encoded.columns]

# Масштабирование признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
test_scaled = scaler.transform(test_encoded)

# Добавление полиномиальных признаков для ключевых числовых колонок
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled[:, numeric_indices])
test_poly = poly.transform(test_scaled[:, numeric_indices])
X_scaled = np.hstack([X_scaled, X_poly])
test_scaled = np.hstack([test_scaled, test_poly])

# Обучение Lasso с кросс-валидацией
lasso = LassoCV(alphas=np.logspace(-4, 0, 50), cv=5, max_iter=10000)
#lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1.0], cv=5)
lasso.fit(X_scaled, y)
lasso_scores = cross_val_score(lasso, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
lasso_rmse = np.sqrt(-lasso_scores)
print(f"Средний RMSE на кросс-валидации (Lasso): {lasso_rmse.mean():.4f}")

y_pred_log = lasso.predict(test_scaled)
y_pred = np.expm1(y_pred_log)

# Сохранение результатов
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

Средний RMSE на кросс-валидации (Lasso): 0.1521
Файл submission.csv успешно сохранен!


### 0.01376 и 0.01347 без логарифмирования на Public, 10,75 на Private
- Добавлены новые фичи
- Убрано логарифмирование признаков с сильной скошенностью


In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score

# Загрузка данных
train_data = pd.read_csv('house-prices-hw/train_hw.csv')
test_data = pd.read_csv('house-prices-hw/test_hw.csv')
test_ids = test_data['Id']
train_data = train_data.drop(columns=['Id'])
test_data = test_data.drop(columns=['Id'])


# Создание новых признаков
def create_new_features(df):
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['HasPool'] = df['PoolArea'] > 0
    df['OverallQualitySF'] = df['OverallQual'] * df['TotalSF']
    df['GarageScore'] = df['GarageCars'] * df['GarageArea']
    df['HasFireplace'] = df['Fireplaces'] > 0
    df['HasGarage'] = df['GarageArea'] > 0
    df['OverallQual_HouseAge'] = df['OverallQual'] * df['HouseAge']
    
    return df

train_data = create_new_features(train_data)
test_data = create_new_features(test_data)

# Определение числовых и категориальных колонок
numeric_cols = train_data.select_dtypes(exclude=['object']).columns.drop('SalePrice')
categorical_cols = train_data.select_dtypes(include=['object']).columns

# Обработка пропусков
def impute_missing_values(df, numeric_cols, categorical_cols):
    imputer = KNNImputer(n_neighbors=5)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    df[categorical_cols] = df[categorical_cols].fillna('None')
    categorical_imputer = SimpleImputer(strategy='most_frequent', fill_value='None')
    df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols])
    return df

train_data = impute_missing_values(train_data, numeric_cols, categorical_cols)
test_data = impute_missing_values(test_data, numeric_cols, categorical_cols)

# Логарифмирование признаков с сильной скошенностью
# skewed_cols = ['TotalSF', 'GarageArea', '1stFlrSF', 'TotalBsmtSF', 'GrLivArea']
# for col in skewed_cols:
#     train_data[col] = np.log1p(train_data[col])
#     test_data[col] = np.log1p(test_data[col])

# Логарифмирование SalePrice
y = np.log1p(train_data['SalePrice'])
X = train_data.drop(columns=['SalePrice'])

# Удаление выбросов по log(SalePrice) с IQR 1.5
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mask = (y >= lower_bound) & (y <= upper_bound)
X = X[mask]
y = y[mask]

# One-Hot Encoding для категориальных признаков
X_encoded = pd.get_dummies(X, drop_first=True)
test_encoded = pd.get_dummies(test_data, drop_first=True)

# Выравнивание колонок
X_encoded, test_encoded = X_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Сохранение индексов ключевых числовых колонок для полиномизации
key_numeric_cols = ['OverallQual', 'TotalSF', 'GrLivArea', 'GarageArea', 'TotalBath']
numeric_indices = [X_encoded.columns.get_loc(col) for col in key_numeric_cols if col in X_encoded.columns]

# Масштабирование признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
test_scaled = scaler.transform(test_encoded)

# Добавление полиномиальных признаков для ключевых числовых колонок
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled[:, numeric_indices])
test_poly = poly.transform(test_scaled[:, numeric_indices])
X_scaled = np.hstack([X_scaled, X_poly])
test_scaled = np.hstack([test_scaled, test_poly])

# Обучение Lasso с кросс-валидацией
lasso = LassoCV(alphas=np.logspace(-4, 0, 100), cv=5, max_iter=10000)
lasso.fit(X_scaled, y)
lasso_scores = cross_val_score(lasso, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
lasso_rmse = np.sqrt(-lasso_scores)
print(f"Средний RMSE на кросс-валидации (Lasso): {lasso_rmse.mean():.4f}")

# Предсказания на тестовом наборе
y_pred_log = lasso.predict(test_scaled)
y_pred = np.expm1(y_pred_log)

# Сохранение результатов
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

Средний RMSE на кросс-валидации (Lasso): 0.1152
Файл submission.csv успешно сохранен!


### 0,01114 - Public, 0,02470 - Private
- Удаление дублирующихся столбцов
- Удаление столбцов с большим количеством NaN
- Новый фичи
- Модифицировано удаление выбросов
- Удаление признаков со слабой вариативностью
- Target Encoding для категориальных признаков с большим количеством категорий




In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score
from scipy.stats import skew

train_data = pd.read_csv('house-prices-hw/train_hw.csv')
test_data = pd.read_csv('house-prices-hw/test_hw.csv')
test_ids = test_data['Id']
train_data = train_data.drop(columns=['Id'])
test_data = test_data.drop(columns=['Id'])

# Удаление дублирующихся столбцов
train_data = train_data.loc[:, ~train_data.columns.duplicated()]
test_data = test_data.loc[:, ~test_data.columns.duplicated()]

# Определение числовых и категориальных колонок
numeric_cols = train_data.select_dtypes(include=[np.number]).columns.drop('SalePrice')
categorical_cols = train_data.select_dtypes(include=['object']).columns

def remove_high_nan_cols(df, numeric_cols, categorical_cols, numeric_threshold=0.90, categorical_threshold=0.85):
    nan_percent = df.isnull().mean()
    cols_to_drop = []
    for col in df.columns:
        if col in numeric_cols and nan_percent[col] >= numeric_threshold:
            cols_to_drop.append(col)
        elif col in categorical_cols and nan_percent[col] >= categorical_threshold:
            cols_to_drop.append(col)
    df = df.drop(columns=cols_to_drop)
    return df, [col for col in numeric_cols if col not in cols_to_drop], [col for col in categorical_cols if col not in cols_to_drop]

train_data, numeric_cols, categorical_cols = remove_high_nan_cols(train_data, numeric_cols, categorical_cols)
test_data = test_data[[col for col in train_data.columns if col != 'SalePrice']]

def create_new_features(df):
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
    df['HasPool'] = df['PoolArea'] > 0
    df['OverallQualitySF'] = df['OverallQual'] * df['TotalSF']
    df['GarageScore'] = df['GarageCars'] * df['GarageArea']
    df['HasFireplace'] = df['Fireplaces'] > 0
    df['HasGarage'] = df['GarageArea'] > 0
    df['OverallQual_HouseAge'] = df['OverallQual'] * df['HouseAge']
    df['TotalPorch'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['ScreenPorch'] + df['3SsnPorch']
    df['TotalBsmtFin'] = df['BsmtFinSF1'] + df['BsmtFinSF2']
    df['YearSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)
    df['LivingAreaRatio'] = df['GrLivArea'] / (df['LotArea'] + 1e-6)
    df['TotalLot'] = df['LotFrontage'] * df['LotArea']
    return df

train_data = create_new_features(train_data)
test_data = create_new_features(test_data)

numeric_cols = train_data.select_dtypes(include=[np.number]).columns.drop('SalePrice')

def impute_missing_values(df, numeric_cols, categorical_cols):
    imputer = KNNImputer(n_neighbors=5)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    df[categorical_cols] = df[categorical_cols].fillna('None')
    categorical_imputer = SimpleImputer(strategy='most_frequent', fill_value='None')
    df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols])
    return df

train_data = impute_missing_values(train_data, numeric_cols, categorical_cols)
test_data = impute_missing_values(test_data, numeric_cols, categorical_cols)

# Логарифмирование SalePrice
y = np.log1p(train_data['SalePrice'])
X = train_data.drop(columns=['SalePrice'])

def remove_outliers(df, y, col, k_y=1.4, k_col=1.4):
    Q1_y = y.quantile(0.25)
    Q3_y = y.quantile(0.75)
    IQR_y = Q3_y - Q1_y
    mask_y = (y >= Q1_y - k_y * IQR_y) & (y <= Q3_y + k_y * IQR_y)
    
    Q1_col = df[col].quantile(0.25)
    Q3_col = df[col].quantile(0.75)
    IQR_col = Q3_col - Q1_col
    mask_col = (df[col] >= Q1_col - k_col * IQR_col) & (df[col] <= Q3_col + k_col * IQR_col)
    
    mask = mask_y & mask_col
    return mask

custom_coefficients = {
    'LotFrontage': (1.2, 1.8),   # (k_y, k_col)
    'LotArea': (1.4, 1.7),
    'BsmtFinSF1': (1.4, 1.4),
    'TotalBsmtSF': (1.3, 1.6),
    'GrLivArea': (1.35, 1.4),
    'GarageArea': (1.4, 1.6),
}

outlier_cols = list(custom_coefficients.keys())
combined_mask = pd.Series(True, index=X.index)

for col in outlier_cols:
    k_y, k_col = custom_coefficients.get(col)
    col_mask = remove_outliers(X, y, col, k_y=k_y, k_col=k_col)
    combined_mask &= col_mask

print('Исходное количество строк:', X.shape[0])
X = X[combined_mask]
y = y[combined_mask]
print('Количество строк после:', X.shape[0])

for col in outlier_cols:
    col_mask = remove_outliers(X, y, col)
    combined_mask &= col_mask  

# Target Encoding для категориальных признаков с большим количеством категорий
def target_encoding(train_df, test_df, col, target):
    temp_df = train_df.copy()
    temp_df['target'] = target
    means = temp_df.groupby(col)['target'].mean()
    train_df[col + '_target'] = train_df[col].map(means)
    test_df[col + '_target'] = test_df[col].map(means).fillna(means.mean())
    return train_df, test_df

categorical_cols_to_encode = categorical_cols.copy()
for col in categorical_cols_to_encode:
    if X[col].nunique() > 10:
        X, test_data = target_encoding(X, test_data, col, y)
        X.drop(columns=[col], inplace=True)
        test_data.drop(columns=[col], inplace=True)
        categorical_cols.remove(col)

# One-Hot Encoding для остальных категориальных признаков
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
test_encoded = pd.get_dummies(test_data, columns=categorical_cols, drop_first=True)

# Выравнивание колонок
X_encoded, test_encoded = X_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Удаление признаков со слабой вариативностью
selector = VarianceThreshold(threshold=0.05)
X_encoded_filtered = selector.fit_transform(X_encoded)
remaining_cols = X_encoded.columns[selector.get_support()]
X_encoded = pd.DataFrame(X_encoded_filtered, columns=remaining_cols, index=X_encoded.index)

test_encoded_filtered = selector.transform(test_encoded)
test_encoded = pd.DataFrame(test_encoded_filtered, columns=remaining_cols, index=test_encoded.index)

# Сохранение индексов ключевых числовых колонок для полиномизации
key_numeric_cols = ['OverallQual', 'TotalSF', 'GrLivArea', 'GarageArea', 'TotalBath']#'Neighborhood'
numeric_indices = [X_encoded.columns.get_loc(col) for col in key_numeric_cols if col in X_encoded.columns]

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_encoded)
test_scaled = scaler.transform(test_encoded)

# Добавление полиномиальных признаков
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled[:, numeric_indices])
test_poly = poly.transform(test_scaled[:, numeric_indices])
X_scaled = np.hstack([X_scaled, X_poly])
test_scaled = np.hstack([test_scaled, test_poly])

# Обучение Lasso с кросс-валидацией
lasso = LassoCV(alphas=np.logspace(-4, 0, 100), cv=5, max_iter=20000)
lasso.fit(X_scaled, y)
lasso_scores = cross_val_score(lasso, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
lasso_rmse = np.sqrt(-lasso_scores)
print(f"Средний RMSE на кросс-валидации (Lasso): {lasso_rmse.mean():.4f}")

y_pred_log = lasso.predict(test_scaled)
y_pred = np.expm1(y_pred_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

### 0.01160 - Public, 0.00922 - Private
#### К лучшему решению:
- Добавлено удаление столбцов с большим количеством NaN
- Добавлены новые фичи

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score

train_data = pd.read_csv('house-prices-hw/train_hw.csv')
test_data = pd.read_csv('house-prices-hw/test_hw.csv')
test_ids = test_data['Id']
train_data = train_data.drop(columns=['Id'])
test_data = test_data.drop(columns=['Id'])

def remove_high_nan_cols(df, numeric_cols, categorical_cols, numeric_threshold=0.70, categorical_threshold=0.80):
    nan_percent = df.isnull().mean()
    cols_to_drop = []
    for col in df.columns:
        if col in numeric_cols and nan_percent[col] >= numeric_threshold:
            cols_to_drop.append(col)
        elif col in categorical_cols and nan_percent[col] >= categorical_threshold:
            cols_to_drop.append(col)
    df = df.drop(columns=cols_to_drop)
    return df, [col for col in numeric_cols if col not in cols_to_drop], [col for col in categorical_cols if col not in cols_to_drop]

def create_new_features(df):
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['HasPool'] = df['PoolArea'] > 0
    df['OverallQualitySF'] = df['OverallQual'] * df['TotalSF']
    df['GarageScore'] = df['GarageCars'] * df['GarageArea']
    df['HasFireplace'] = df['Fireplaces'] > 0
    df['HasGarage'] = df['GarageArea'] > 0
    df['OverallQual_HouseAge'] = df['OverallQual'] * df['HouseAge']
    df['TotalPorch'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['ScreenPorch'] + df['3SsnPorch']
    df['TotalBsmtFin'] = df['BsmtFinSF1'] + df['BsmtFinSF2']
    df['YearSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)
    df['LivingAreaRatio'] = df['GrLivArea'] / (df['LotArea'] + 1e-6)
    df['TotalLot'] = df['LotFrontage'] * df['LotArea']
    return df

# Определение числовых и категориальных колонок
numeric_cols = train_data.select_dtypes(exclude=['object']).columns.drop('SalePrice')
categorical_cols = train_data.select_dtypes(include=['object']).columns

train_data = create_new_features(train_data)
test_data = create_new_features(test_data)

train_data, numeric_cols, categorical_cols = remove_high_nan_cols(train_data, numeric_cols, categorical_cols)
test_data = test_data[[col for col in train_data.columns if col != 'SalePrice']]

# Определение числовых и категориальных колонок
numeric_cols = train_data.select_dtypes(exclude=['object']).columns.drop('SalePrice')
categorical_cols = train_data.select_dtypes(include=['object']).columns

def impute_missing_values(df, numeric_cols, categorical_cols):
    imputer = KNNImputer(n_neighbors=5)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    df[categorical_cols] = df[categorical_cols].fillna('None')
    categorical_imputer = SimpleImputer(strategy='most_frequent', fill_value='None')
    df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols])
    return df

train_data = impute_missing_values(train_data, numeric_cols, categorical_cols)
test_data = impute_missing_values(test_data, numeric_cols, categorical_cols)

y = np.log1p(train_data['SalePrice'])
X = train_data.drop(columns=['SalePrice'])

# Удаление выбросов по log(SalePrice) с IQR 1.5
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mask = (y >= lower_bound) & (y <= upper_bound)
X = X[mask]
y = y[mask]

# One-Hot Encoding для категориальных признаков
X_encoded = pd.get_dummies(X, drop_first=True)
test_encoded = pd.get_dummies(test_data, drop_first=True)

# Выравнивание колонок
X_encoded, test_encoded = X_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Сохранение индексов ключевых числовых колонок для полиномизации
key_numeric_cols = ['OverallQual', 'TotalSF', 'GrLivArea', 'GarageArea', 'TotalBath']
numeric_indices = [X_encoded.columns.get_loc(col) for col in key_numeric_cols if col in X_encoded.columns]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
test_scaled = scaler.transform(test_encoded)

# Добавление полиномиальных признаков для ключевых числовых колонок
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled[:, numeric_indices])
test_poly = poly.transform(test_scaled[:, numeric_indices])
X_scaled = np.hstack([X_scaled, X_poly])
test_scaled = np.hstack([test_scaled, test_poly])

# Обучение Lasso с кросс-валидацией
lasso = LassoCV(alphas=np.logspace(-4, 0, 100), cv=5, max_iter=10000)
lasso.fit(X_scaled, y)
lasso_scores = cross_val_score(lasso, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
lasso_rmse = np.sqrt(-lasso_scores)
print(f"Средний RMSE на кросс-валидации (Lasso): {lasso_rmse.mean():.4f}")

y_pred_log = lasso.predict(test_scaled)
y_pred = np.expm1(y_pred_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

Средний RMSE на кросс-валидации (Lasso): 0.1143
Файл submission.csv успешно сохранен!
